In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/customer_analytics.csv"

Mounted at /content/drive


In [2]:
df = pd.read_csv(FILE_PATH)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(df.shape)

(779495, 9)


Recency Date

In [3]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print(snapshot_date)

2011-12-10 12:50:00


In [4]:
rfm = (
    df.groupby("Customer ID")
      .agg({
          "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
          "Invoice": "nunique",
          "Revenue": "sum"
      })
      .reset_index()
)

rfm.columns = [
    "Customer ID",
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [5]:
rfm.describe()

,Customer ID,Recency,Frequency,Monetary
count,5881.000000,5881.000000,5881.000000,5881.000000
mean,15314.674205,201.457745,6.287196,2954.396237
std,1715.429759,209.474135,13.012879,14437.322635
min,12346.000000,1.000000,1.000000,0.000000
25%,13833.000000,26.000000,1.000000,341.900000
50%,15313.000000,96.000000,3.000000,865.600000
75%,16797.000000,380.000000,7.000000,2247.720000
max,18287.000000,739.000000,398.000000,580987.040000


R Score

In [6]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

F Score

In [7]:
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

M Score

In [8]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

In [9]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346.0,326,12,77556.46,2,5,5
1,12347.0,2,8,4921.53,5,4,5
2,12348.0,75,5,2019.40,3,4,4
3,12349.0,19,4,4428.69,5,3,5
4,12350.0,310,1,334.40,2,1,2


RFM Score

In [10]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    +
    rfm["F_Score"].astype(str)
    +
    rfm["M_Score"].astype(str)
)

In [11]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,12,77556.46,2,5,5,255
1,12347.0,2,8,4921.53,5,4,5,545
2,12348.0,75,5,2019.40,3,4,4,344
3,12349.0,19,4,4428.69,5,3,5,535
4,12350.0,310,1,334.40,2,1,2,212


In [12]:
rfm["RFM_Score"].value_counts().head(20)

,count
RFM_Score,
555,469
111,318
455,248
121,169
211,165
344,164
112,148
444,147
355,131


Segmentation

In [13]:
def segment_customer(row):

    r = int(row["R_Score"])
    f = int(row["F_Score"])

    if r >= 4 and f >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f >= 2:
        return "Potential Loyalists"

    elif r >= 3 and f >= 2:
        return "Need Attention"

    elif r <= 2 and f >= 3:
        return "At Risk"

    else:
        return "Lost Customers"

In [14]:
rfm["Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

In [15]:
rfm["Segment"].value_counts()

,count
Segment,
Lost Customers,1885
Champions,1482
At Risk,824
Potential Loyalists,714
Loyal Customers,516
Need Attention,460


Revenue By Segment

In [16]:
segment_revenue = (
    rfm.groupby("Segment")["Monetary"]
       .sum()
       .sort_values(ascending=False)
)

print(segment_revenue)

Segment
Champions              1.202433e+07
Loyal Customers        1.691928e+06
At Risk                1.589384e+06
Potential Loyalists    8.158920e+05
Lost Customers         7.924799e+05
Need Attention         4.607895e+05
Name: Monetary, dtype: float64


Customer Count By Segment

In [17]:
segment_count = (
    rfm["Segment"]
      .value_counts()
)

print(segment_count)

Segment
Lost Customers         1885
Champions              1482
At Risk                 824
Potential Loyalists     714
Loyal Customers         516
Need Attention          460
Name: count, dtype: int64


In [18]:
rfm.to_csv(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/rfm_customers.csv",
    index=False
)

print("RFM Dataset Saved")

RFM Dataset Saved


In [19]:
rfm_summary = (
    rfm.groupby("Segment")
       .agg({
           "Customer ID":"count",
           "Monetary":"sum",
           "Frequency":"mean",
           "Recency":"mean"
       })
       .round(2)
)

rfm_summary

,Customer ID,Monetary,Frequency,Recency
Segment,,,,
At Risk,824,1589384.31,4.94,369.10
Champions,1482,12024330.14,15.66,20.37
Lost Customers,1885,792479.94,1.20,385.07
Loyal Customers,516,1691928.38,8.54,106.29
Need Attention,460,460789.45,2.55,111.63
Potential Loyalists,714,815892.04,2.59,25.76
